# Smoke test: LLaVA-Med-v1.5-7B as an answerer

**Environment: **path-llava** - NOT the main environment**

Validates this model in the answerer harness before committing it to a multi-hour
production run across three datasets and five conditions.

LLaVA-Med is the **canonical biomedical baseline** - the model reviewers will
expect to see, and the one PathMMU and PathVQA papers benchmark against.

Two things make it awkward, both measured rather than assumed:

**It runs in a separate conda environment.** Its `LlavaMistralForCausalLM` class does not
exist in transformers, and no HF-format conversion has been published. The official
`llava-torch` package supplies it but pins `torch==2.0.1` / `transformers==4.36.2`, which
would downgrade the stack the judges and four other answerers depend on. See
`../ENVIRONMENT.md`.

**It ignores the output format.** It answers in prose rather than a letter, so a short
budget truncates before the answer appears. On 12 PathOPEN MCQs: 2/12 parsed at 16 tokens,
7/12 at 96. Adding the `token_overlap` parsing rule lifted this to 24/30 - the model was
answering correctly in its own words, and exact string matching could not see it.

**It confabulates when blinded.** With no image it still describes "a histological section
of a tumor", where MedGemma correctly reports an abstract painting. The blind condition is
still valid - no image is passed - but here it measures text priors *plus* confabulation.

## What is checked

1. Loads at the expected precision on one GPU, **no CPU offload** - `device_map` silently
   offloads when a model does not fit, turning an hours-long run into a days-long one.
2. **Describes a real slide** - the cheapest way to detect a broken vision path, which
   still emits fluent text but stops being about the image.
3. **Answers MCQs from all three datasets**, parsed correctly.
4. **The image matters** (blind condition, sub-pillar 3b).
5. **Option order does not** (position-shuffle, sub-pillar 3b).
6. Throughput, so the full run can be costed rather than guessed at.

The check logic lives in `smoke_common.py` so every model is measured identically -
if the checks drifted between notebooks, cross-model numbers would stop being
comparable, and comparability is the entire point of sub-pillar 4a.

In [ ]:
import os
import sys

# Pick the GPU BEFORE torch initializes CUDA. GPUs 0-2 are busy with the Qwen judge's
# Benchmark 5 run in ../../vlm/.
os.environ["CUDA_VISIBLE_DEVICES"] = "5"

sys.path.insert(0, os.path.abspath(".."))
sys.path.insert(0, os.path.abspath("."))

import collections

import torch

import smoke_common as sc

MODEL_KEY = "llavamed"
MAX_NEW_TOKENS = 96   # measured; see the note above
N_PER_DATASET = 30   # 30, not 20: at smaller n a run of identical letters looks like
                     # positional bias when it is only sampling noise
print(torch.__version__, "|", torch.cuda.device_count(), "visible GPU(s)")

## 1. Load, and verify where the weights landed

In [ ]:
import time

t0 = time.time()
from answerer_llava import load_llava_answerer
model = load_llava_answerer(MODEL_KEY, device="cuda:0")
print(f"loaded in {time.time() - t0:.1f}s")

devices = collections.Counter(str(p.device) for p in model.model.parameters())
offloaded = [d for d in devices if d in ("cpu", "meta", "disk")]
print(f"parameter devices: {dict(devices)}")
print(f"memory: {model.memory_summary()}")
assert not offloaded, f"model offloaded to {offloaded} - it will be unusably slow"
print("OK: fully resident on GPU, no offload")

## 2. Does it see the image?

Two different slides, then no slide at all. A and B **must differ** - identical
descriptions mean the vision path is dead. The no-image reply is diagnostic in its own
right: some models notice the absence, others confabulate a plausible histology
description. Both are informative; neither is a bug.

In [ ]:
tasks_pathopen = sc.load_tasks("PathOPEN", N_PER_DATASET)
print(f"{len(tasks_pathopen)} PathOPEN tasks loaded\n")
sc.check_vision(model, tasks_pathopen)

In [ ]:
from PIL import Image

# Look at the slide yourself. No automated check distinguishes a plausible description
# from an accurate one - this is where a pathologist's eye is worth more than an assert.
Image.open(tasks_pathopen[0]["image_path"]).convert("RGB").resize((512, 512))

## 3. MCQ answering across all three datasets

Each dataset stores options differently (PathOPEN in separate columns; PathMMU and
PatchVQA as letter-prefixed lists with 2-16 options), so all three are exercised rather
than assuming one generalises.

Read `parse_rate` alongside accuracy. A low parse rate is a **format** problem, not a
knowledge problem, and conflating them would turn a formatting difference into a fake
accuracy gap - exactly the artifact Pillar 4's cross-model ranking must not contain.

In [ ]:
task_sets = {
    "PathOPEN": tasks_pathopen,
    "PathMMU": sc.load_tasks("PathMMU", N_PER_DATASET),
    "PatchVQA": sc.load_tasks("PatchVQA", N_PER_DATASET),
}

full_results, per_item_seconds = {}, {}
for name, task_list in task_sets.items():
    print(f"\n=== {name} ({len(task_list)} items) ===")
    records, seconds = sc.run_condition(model, task_list,
                                        max_new_tokens=MAX_NEW_TOKENS, show=3)
    full_results[name] = records
    per_item_seconds[name] = seconds
    sc.report(records, name, seconds)

## 4. Does the image matter? (blind condition, sub-pillar 3b)

Same questions, image withheld.

A small gap has two very different causes, and they must not be confused: the questions
are answerable from text alone (a real dataset finding, and what 3b exists to quantify),
or the image never reached the model (a harness bug invalidating Pillars 3b and 4).
Section 2 settles which, which is why it runs first.

**At n=30 this diagnostic is weak.** MedGemma measured a 0pp gap here while genuinely
using the image - 25/30 predictions were identical between conditions and the 5 that
flipped cancelled out. Treat anything under ~5pp as uninformative at this sample size;
the production run uses all 1,428 tasks for exactly this reason.

In [ ]:
blind_stats = sc.blind_comparison(model, task_sets, full_results,
                                  max_new_tokens=MAX_NEW_TOKENS)

## 5. Does option order matter? (position-shuffle, sub-pillar 3b)

Three permutations of the same questions. A model reading the options scores about the
same each time; one keying on position swings.

Compare `predicted` against `truth`, not `predicted` alone - raw accuracy can hide bias
when ground truth happens to be spread the same way.

In [ ]:
shuffle_stats = sc.shuffle_comparison(model, task_sets,
                                      max_new_tokens=MAX_NEW_TOKENS)

## 6. Verdict and cost projection

In [ ]:
sc.cost_projection(sum(per_item_seconds.values()) / len(per_item_seconds))
print(sc.CHECKLIST)

In [ ]:
# Free the GPU for the next smoke test.
del model
torch.cuda.empty_cache()
print("released")